# CrisisConnect: training the crisis classifiers

Fine-tunes two classifiers from [`crisistransformers/CT-M1-Complete`](https://huggingface.co/crisistransformers/CT-M1-Complete):

| Model | Task | Trained on | Tested on |
|---|---|---|---|
| **Binary** | crisis vs. non-crisis | project tweets, [Disaster Tweets (2020)](https://www.kaggle.com/datasets/vstepanenko/disaster-tweets), TREC-IS | held-out splits, unseen TREC-IS events, and **[NLP with Disaster Tweets (2015)](https://www.kaggle.com/competitions/nlp-getting-started)**, a separate dataset never used for training or model selection |
| **Priority** | Low / Medium / High / Critical | [TREC Incident Streams](https://www.dcs.gla.ac.uk/~richardm/TREC_IS/) (~100k human-labelled crisis tweets, 2012-2021) | unseen TREC-IS events, and the project's Kenya-focused tweets |

**Priority** follows TREC-IS: *how urgently an emergency manager needs to see this information*, not how big the
event is. A tornado warning for named towns is Critical; a news story mentioning last week's tornado is Low.
It outputs the same four labels as the earlier severity model, so the app can use it unchanged.

Evaluation design:

- **Unseen events.** TREC-IS tweets are split by *event*, not at random: tweets about one disaster are
  near-duplicates, so a random split would leak. Both models hold out the same events.
- **Separate validation split** for choosing the best epoch; test data is never used for selection.
- **No overlap.** Texts are compared after normalization (case, URLs, mentions, punctuation) and every test set is
  cleaned of anything that appears in training.
- **Every metric is written to `output/metrics.json`**; model cards with the results are generated.

**Run on Colab with a GPU** (Runtime → Change runtime type → T4 GPU). The whole notebook takes about 30 minutes.

**Data files** (uploaded when prompted, or placed in `data/`):
- `crisis_tweets.xlsx`: project tweets (sheet `new 1`)
- `booster_severity.csv`: extra severity-labelled project tweets
- `tweets.csv`: the Kaggle *Disaster Tweets* dataset (`id, keyword, location, text, target`)

TREC-IS, the external test set and CrisisBench are downloaded at run time. They contain tweet text released for
research, so they are not stored in this repository.

In [ ]:
# --- Configuration ---
SMOKE_TEST = False          # True: tiny CPU run that exercises every cell (for testing the notebook itself)
SEED = 42
BASE_MODEL = "crisistransformers/CT-M1-Complete"
MAX_LENGTH = 128            # the base model has 130 positions; RoBERTa reserves 2
DATA_DIR = "data"
OUTPUT_DIR = "output"
PREVIOUS_MODELS = {"binary": "ron4444444/crisis-binary-model", "priority": "ron4444444/crisis-severity-model"}

BINARY = dict(epochs=3, lr=2e-5, batch_size=32)
PRIORITY = dict(epochs=3, lr=2e-5, batch_size=32)
# TREC-IS adds hard negatives to the binary model; capped so it doesn't outweigh the other sources
TRECIS_BINARY_TRAIN_CAP = 20_000
TRECIS_BINARY_VAL_CAP = 4_000

TRECIS_URLS = [
    "http://dcs.gla.ac.uk/~richardm/TREC_IS/2021/TRECIS-2018-2020B.json.gz",
    "http://dcs.gla.ac.uk/~richardm/TREC_IS/2021/TRECIS-2021A-2021B.json.gz",
]
EXTERNAL_TEST_URL = ("https://huggingface.co/datasets/venetis/disaster_tweets/resolve/"
                     "refs%2Fconvert%2Fparquet/default/train/0000.parquet")
CRISISBENCH_URL = ("https://huggingface.co/datasets/QCRI/CrisisBench-english/resolve/"
                   "refs%2Fconvert%2Fparquet/informativeness/test/0000.parquet")

In [ ]:
%pip install -q "transformers>=4.46" accelerate scikit-learn openpyxl pyarrow

In [ ]:
import gzip
import json
import os
import random
import re
import shutil
import urllib.request

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
import transformers
from transformers import (AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding,
                          Trainer, TrainerCallback, TrainingArguments, set_seed)

transformers.logging.set_verbosity_error()   # hides the expected "classifier weights newly initialized" report

set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")

if SMOKE_TEST:
    BINARY.update(epochs=1, batch_size=8)
    PRIORITY.update(epochs=1, batch_size=8)
    TRECIS_BINARY_TRAIN_CAP, TRECIS_BINARY_VAL_CAP = 200, 60

try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
REQUIRED = ["crisis_tweets.xlsx", "booster_severity.csv", "tweets.csv"]
missing = [f for f in REQUIRED if not os.path.exists(os.path.join(DATA_DIR, f))]
if missing and IN_COLAB:
    print("Upload these files:", missing)
    for name, content in colab_files.upload().items():
        with open(os.path.join(DATA_DIR, name), "wb") as f:
            f.write(content)
missing = [f for f in REQUIRED if not os.path.exists(os.path.join(DATA_DIR, f))]
assert not missing, f"Missing data files in {DATA_DIR}/: {missing}"

## Data

Duplicates (after normalization) with conflicting labels are dropped from the project and Kaggle data. TREC-IS
has multiple annotations for some tweets; those take the majority label, and ties are dropped.

In [ ]:
PRIORITY_LABELS = ["Low", "Medium", "High", "Critical"]   # ordinal; same names as the deployed severity model


def normalize(text):
    text = re.sub(r"https?://\S+|@\w+|^rt\s+", " ", str(text).lower())
    return re.sub(r"[^a-z0-9]+", " ", text).strip()


def is_true(value):
    return str(value).strip().lower() == "true"


def dedupe(df):
    """Drops empty texts, keeps one copy of consistent duplicates, drops conflicting ones."""
    df = df[df.text.notna() & (df.text.astype(str).str.strip() != "")].copy()
    df["norm"] = df.text.map(normalize)
    conflicting = df.groupby("norm").label.nunique()
    df = df[~df.norm.isin(conflicting[conflicting > 1].index)]
    return df.drop_duplicates("norm").reset_index(drop=True)


# --- Project + Kaggle 2020 (binary only) ---
xlsx = pd.read_excel(os.path.join(DATA_DIR, "crisis_tweets.xlsx"), sheet_name="new 1")
booster = pd.read_csv(os.path.join(DATA_DIR, "booster_severity.csv"))
project = pd.concat([xlsx, booster], ignore_index=True)
project = project[project.crisis_detection.notna()]   # booster rows without a flag are unlabelled, not negatives
project = pd.DataFrame({"text": project.tweet_text, "label": project.crisis_detection.map(is_true).astype(int),
                        "severity": project.severity_score, "source": "project"})

kaggle_2020 = pd.read_csv(os.path.join(DATA_DIR, "tweets.csv"))
kaggle_2020 = pd.DataFrame({"text": kaggle_2020.text, "label": kaggle_2020.target.astype(int),
                            "severity": np.nan, "source": "kaggle_2020"})
pool = dedupe(pd.concat([project, kaggle_2020], ignore_index=True))

# --- External binary test set and CrisisBench ---
external = pd.read_parquet(EXTERNAL_TEST_URL)
external = dedupe(pd.DataFrame({"text": external.text, "label": external.target.astype(int)}))
crisisbench = pd.read_parquet(CRISISBENCH_URL).sample(n=3000, random_state=SEED)
crisisbench = pd.DataFrame({"text": crisisbench.text, "label": (crisisbench.class_label == "informative").astype(int)})

# --- TREC-IS ---
rows = []
for url in TRECIS_URLS:
    with urllib.request.urlopen(url) as response:
        data = json.loads(gzip.decompress(response.read()))
    for event in data["events"]:
        for t in event["tweets"]:
            rows.append({"text": t.get("postText"), "event": t.get("eventID") or event["eventid"],
                         "type": event.get("type") or "unknown", "priority": t.get("postPriority"),
                         "irrelevant": "Irrelevant" in (t.get("postCategories") or [])})
trecis = pd.DataFrame(rows)
trecis = trecis[trecis.text.notna() & (trecis.text.str.strip() != "") & trecis.priority.isin(PRIORITY_LABELS)]
if SMOKE_TEST:
    trecis = trecis.sample(4000, random_state=SEED)
trecis["norm"] = trecis.text.map(normalize)
trecis["priority_id"] = trecis.priority.map(PRIORITY_LABELS.index)


def majority(values):
    counts = values.value_counts()
    return counts.index[0] if len(counts) == 1 or counts.iloc[0] > counts.iloc[1] else np.nan


trecis = (trecis.groupby("norm")
          .agg(text=("text", "first"), event=("event", "first"), type=("type", "first"),
               priority_id=("priority_id", majority), irrelevant=("irrelevant", majority))
          .dropna(subset=["priority_id", "irrelevant"]).reset_index())
trecis["priority_id"] = trecis.priority_id.astype(int)
trecis["irrelevant"] = trecis.irrelevant.astype(bool)

# Each text lives in exactly one source, so nothing can leak between splits of different sources
trecis = trecis[~trecis.norm.isin(set(pool.norm))]
training_texts = set(pool.norm) | set(trecis.norm)
for name in ("external", "crisisbench"):
    df = globals()[name]
    df["norm"] = df.text.map(normalize)
    overlap = df.norm.isin(training_texts)
    print(f"{name}: removed {overlap.sum()} tweets that also appear in the training data")
    globals()[name] = df[~overlap].reset_index(drop=True)

if SMOKE_TEST:
    pool = pd.concat([g.sample(min(len(g), 60), random_state=SEED) for _, g in pool.groupby(["source", "label"])])
    external, crisisbench = external.sample(100, random_state=SEED), crisisbench.sample(100, random_state=SEED)

print(pool.groupby(["source", "label"]).size().rename("tweets").to_frame())
print(f"TREC-IS: {len(trecis)} unique tweets from {trecis.event.nunique()} events, {trecis.irrelevant.sum()} irrelevant")
print(f"external test: {len(external)} tweets, {external.label.mean():.1%} crisis")

In [ ]:
# --- TREC-IS: split by event, cycling through events within each disaster type so every split covers most types
rng = np.random.default_rng(SEED)
event_split = {}
for _, events in trecis.groupby("type").event.unique().items():
    events = list(events)
    rng.shuffle(events)
    for i, event in enumerate(events):
        event_split[event] = ["test", "val", "train", "train", "train", "train"][i % 6] if len(events) >= 3 else "train"
trecis["split"] = trecis.event.map(event_split)
print(trecis.groupby("split").agg(events=("event", "nunique"), tweets=("text", "size")))
print("held-out test events:", sorted(trecis[trecis.split == "test"].event.unique()))

# --- Binary data
# TREC-IS "Irrelevant" -> Non-Crisis; relevant Medium/High/Critical -> Crisis.
# Relevant Low-priority tweets are ambiguous for a crisis/non-crisis decision and are left out.
trecis_binary = trecis[trecis.irrelevant | (trecis.priority_id >= 1)].assign(
    label=lambda d: (~d.irrelevant).astype(int), source="trecis")


def balanced_sample(df, n):
    return pd.concat([g.sample(min(len(g), n // 2), random_state=SEED) for _, g in df.groupby("label")])


strata = pool.source + "_" + pool.label.astype(str)
pool_train, rest = train_test_split(pool, test_size=0.30, stratify=strata, random_state=SEED)
pool_val, pool_test = train_test_split(rest, test_size=0.50, stratify=strata[rest.index], random_state=SEED)
by_split = {s: trecis_binary[trecis_binary.split == s] for s in ("train", "val", "test")}
train_df = pd.concat([pool_train, balanced_sample(by_split["train"], TRECIS_BINARY_TRAIN_CAP)], ignore_index=True)
val_df = pd.concat([pool_val, balanced_sample(by_split["val"], TRECIS_BINARY_VAL_CAP)], ignore_index=True)
test_df = pd.concat([pool_test, by_split["test"].sample(min(len(by_split["test"]), 60), random_state=SEED)
                     if SMOKE_TEST else by_split["test"]], ignore_index=True)
for name, split in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"binary {name:5} {len(split):6} tweets  ({split.label.mean():.1%} crisis)", split.source.value_counts().to_dict())
    split[["text", "label", "source"]].to_csv(os.path.join(OUTPUT_DIR, f"binary_{name}.csv"), index=False)
external[["text", "label"]].to_csv(os.path.join(OUTPUT_DIR, "binary_external_test.csv"), index=False)

## Training helpers

Class weights are passed to each trainer (the first version used a global variable, so whichever model was
trained last silently changed the weights). The best epoch's weights are kept in memory rather than saved as
checkpoints, because writing a checkpoint to Colab's disk took longer than training an epoch.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.model_max_length = MAX_LENGTH


class TextDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels):
        self.encodings = tokenizer(list(texts), truncation=True, max_length=MAX_LENGTH)
        self.labels = list(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.encodings.items()}
        item["labels"] = int(self.labels[i])
        return item


class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        weight = self.class_weights.to(outputs.logits.device)
        loss = torch.nn.functional.cross_entropy(outputs.logits, labels, weight=weight)
        return (loss, outputs) if return_outputs else loss


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=1)
    return {"accuracy": accuracy_score(labels, preds), "f1_macro": f1_score(labels, preds, average="macro")}


class KeepBestInMemory(TrainerCallback):
    """Keeps the weights of the epoch with the best validation macro F1 in RAM."""

    def __init__(self):
        self.best_f1, self.best_epoch, self.weights = -1.0, None, None

    def on_evaluate(self, args, state, control, metrics=None, model=None, **kwargs):
        if metrics and metrics.get("eval_f1_macro", -1) > self.best_f1:
            self.best_f1, self.best_epoch = metrics["eval_f1_macro"], round(state.epoch)
            self.weights = {k: v.detach().to("cpu", copy=True) for k, v in model.state_dict().items()}


def train_classifier(train, val, labels, out_dir, epochs, lr, batch_size):
    """Fine-tunes BASE_MODEL and returns (model, best_epoch, best validation metrics)."""
    model = AutoModelForSequenceClassification.from_pretrained(
        BASE_MODEL, num_labels=len(labels),
        id2label=dict(enumerate(labels)), label2id={l: i for i, l in enumerate(labels)})
    weights = compute_class_weight("balanced", classes=np.arange(len(labels)), y=train.label.to_numpy())
    total_steps = int(np.ceil(len(train) / batch_size)) * epochs
    args = TrainingArguments(
        output_dir=out_dir, num_train_epochs=epochs, learning_rate=lr,
        per_device_train_batch_size=batch_size, per_device_eval_batch_size=128,
        warmup_steps=int(0.1 * total_steps),   # 10% warmup; warmup_ratio was removed in transformers v5
        weight_decay=0.01, seed=SEED, report_to="none",
        fp16=DEVICE == "cuda", logging_strategy="epoch", eval_strategy="epoch", save_strategy="no",
    )
    keep_best = KeepBestInMemory()
    trainer = WeightedTrainer(
        model=model, args=args, processing_class=tokenizer, data_collator=DataCollatorWithPadding(tokenizer),
        train_dataset=TextDataset(train.text, train.label), eval_dataset=TextDataset(val.text, val.label),
        compute_metrics=compute_metrics, class_weights=torch.tensor(weights, dtype=torch.float),
        callbacks=[keep_best],
    )
    trainer.train()
    trainer.model.load_state_dict(keep_best.weights)
    return trainer.model, keep_best.best_epoch, round(keep_best.best_f1, 4)


@torch.no_grad()
def predict(model, texts, tok=tokenizer, batch_size=128):
    model.eval().to(DEVICE)
    out = []
    texts = list(texts)
    for i in range(0, len(texts), batch_size):
        enc = tok(texts[i:i + batch_size], truncation=True, max_length=MAX_LENGTH, padding=True, return_tensors="pt").to(DEVICE)
        out.append(model(**enc).logits.float().cpu().numpy())
    return np.concatenate(out).argmax(axis=1)


def binary_metrics(truth, pred):
    truth, pred = np.asarray(truth), np.asarray(pred)
    return {
        "n": int(len(truth)),
        "accuracy": round(accuracy_score(truth, pred), 4),
        "precision_crisis": round(precision_score(truth, pred, zero_division=0), 4),
        "recall_crisis": round(recall_score(truth, pred, zero_division=0), 4),
        "f1_crisis": round(f1_score(truth, pred, zero_division=0), 4),
        "f1_macro": round(f1_score(truth, pred, average="macro"), 4),
        "confusion_[[tn,fp],[fn,tp]]": confusion_matrix(truth, pred, labels=[0, 1]).tolist(),
    }


def priority_metrics(truth, pred):
    truth, pred = np.asarray(truth), np.asarray(pred)
    urgent_truth, urgent_pred = truth >= 2, pred >= 2   # High or Critical: needs prompt attention
    return {
        "n": int(len(truth)),
        "accuracy": round(accuracy_score(truth, pred), 4),
        "f1_macro": round(f1_score(truth, pred, average="macro", labels=range(4), zero_division=0), 4),
        "recall_per_class": dict(zip(PRIORITY_LABELS, np.round(recall_score(truth, pred, average=None, labels=range(4), zero_division=0), 4).tolist())),
        "off_by_2plus": round(float((np.abs(pred - truth) >= 2).mean()), 4),
        "urgent_precision": round(precision_score(urgent_truth, urgent_pred, zero_division=0), 4),
        "urgent_recall": round(recall_score(urgent_truth, urgent_pred, zero_division=0), 4),
        "confusion_rows_true_cols_pred": confusion_matrix(truth, pred, labels=range(4)).tolist(),
    }


def table(results):
    return pd.DataFrame({k: {m: v for m, v in r.items() if not isinstance(v, (list, dict))} for k, r in results.items()}).T


metrics = {"seed": SEED, "base_model": BASE_MODEL, "smoke_test": SMOKE_TEST,
           "trecis_test_events": sorted(trecis[trecis.split == "test"].event.unique())}

## Binary model

In [ ]:
BINARY_LABELS = ["Non-Crisis", "Crisis"]   # same label names as the deployed model
binary_model, best_epoch, best_val = train_classifier(train_df, val_df, BINARY_LABELS, "checkpoints/binary", **BINARY)
print(f"best epoch {best_epoch}, validation macro F1 {best_val}")

test_sets = {
    "test_project": test_df[test_df.source == "project"],
    "test_kaggle_2020": test_df[test_df.source == "kaggle_2020"],
    "test_trecis_unseen_events": test_df[test_df.source == "trecis"],
    "external_kaggle_2015": external,
    "crisisbench_informativeness_proxy": crisisbench,
}
predictions = {name: predict(binary_model, data.text) for name, data in test_sets.items()}
metrics["binary"] = {"train_size": len(train_df), "best_epoch": best_epoch, "best_val_f1_macro": best_val,
                     **{name: binary_metrics(data.label, predictions[name]) for name, data in test_sets.items()}}

# The previous model on the same data. Its training data is unknown, so only sets it cannot have seen are compared.
previous = AutoModelForSequenceClassification.from_pretrained(PREVIOUS_MODELS["binary"])
previous_tok = AutoTokenizer.from_pretrained(PREVIOUS_MODELS["binary"])
metrics["previous_binary_model"] = {
    name: binary_metrics(test_sets[name].label, predict(previous, test_sets[name].text, tok=previous_tok))
    for name in ("external_kaggle_2015", "test_trecis_unseen_events", "crisisbench_informativeness_proxy")
}
del previous

print(table({k: v for k, v in metrics["binary"].items() if isinstance(v, dict)}))
print("\nPrevious model:")
print(table(metrics["previous_binary_model"]))

In [ ]:
# Examples of external-test errors, to read rather than just count
ext = external.assign(pred=predictions["external_kaggle_2015"])
errors = pd.concat([ext[(ext.label == 1) & (ext.pred == 0)].head(15).assign(error="missed crisis"),
                    ext[(ext.label == 0) & (ext.pred == 1)].head(15).assign(error="false alarm")])
errors[["error", "text"]].to_csv(os.path.join(OUTPUT_DIR, "binary_external_errors.csv"), index=False)
pd.set_option("display.max_colwidth", 140)
errors[["error", "text"]]

## Priority model

Trained on relevant TREC-IS tweets (the ones not tagged *Irrelevant*), with the best epoch chosen on held-out
validation events, and tested on other held-out events.

It is also tested on the project's own crisis tweets. Those carry *severity* labels (how big the event is), not
TREC-IS priority, so that score measures how well the two notions line up on Kenyan community reports rather
than plain accuracy.

In [ ]:
relevant = trecis[~trecis.irrelevant].assign(label=lambda d: d.priority_id)
p_train, p_val, p_test = (relevant[relevant.split == s] for s in ("train", "val", "test"))
if SMOKE_TEST:
    p_train, p_val, p_test = (d.sample(min(len(d), n), random_state=SEED) for d, n in ((p_train, 200), (p_val, 60), (p_test, 60)))
for name, split in [("train", p_train), ("val", p_val), ("test", p_test)]:
    print(f"priority {name:5} {len(split):6} tweets", split.priority_id.map(dict(enumerate(PRIORITY_LABELS))).value_counts().reindex(PRIORITY_LABELS).to_dict())

priority_model, best_epoch, best_val = train_classifier(p_train, p_val, PRIORITY_LABELS, "checkpoints/priority", **PRIORITY)
print(f"best epoch {best_epoch}, validation macro F1 {best_val}")

project_crisis = dedupe(project[(project.label == 1) & project.severity.isin(PRIORITY_LABELS)]
                        .assign(label=lambda d: d.severity.map(PRIORITY_LABELS.index)))
priority_tests = {"test_trecis_unseen_events": p_test, "project_severity_labels": project_crisis}
metrics["priority"] = {"train_size": len(p_train), "best_epoch": best_epoch, "best_val_f1_macro": best_val,
                       **{name: priority_metrics(data.label, predict(priority_model, data.text))
                          for name, data in priority_tests.items()}}

# The previous severity model has not seen TREC-IS, so that comparison is fair (it was trained on the project tweets)
previous = AutoModelForSequenceClassification.from_pretrained(PREVIOUS_MODELS["priority"])
previous_tok = AutoTokenizer.from_pretrained(PREVIOUS_MODELS["priority"])
order = [previous.config.label2id[l] for l in PRIORITY_LABELS]   # its label ids are in alphabetical order
remap = {old: new for new, old in enumerate(order)}
metrics["previous_severity_model"] = {
    "test_trecis_unseen_events": priority_metrics(
        p_test.label, np.vectorize(remap.get)(predict(previous, p_test.text, tok=previous_tok)))}
del previous

print(table({k: v for k, v in metrics["priority"].items() if isinstance(v, dict)}))
print("\nPrevious severity model:")
print(table(metrics["previous_severity_model"]))
print("\nConfusion on unseen events (rows: true, columns: predicted):")
print(pd.DataFrame(metrics["priority"]["test_trecis_unseen_events"]["confusion_rows_true_cols_pred"],
                   index=PRIORITY_LABELS, columns=PRIORITY_LABELS))

## Save

Writes both models (with tokenizers and generated model cards) and `metrics.json` to `output/`.

In [ ]:
def model_card(title, body):
    return f"""---
license: mit
language: en
base_model: {BASE_MODEL}
pipeline_tag: text-classification
---

# {title}

{body}

Trained with [`train_crisis_models.ipynb`](https://github.com/isaac-ron/crisisconnectmodels), seed {SEED}.
Full metrics: `metrics.json`.
"""


def row(label, m):
    return f"| {label} | {m['n']} | {m['accuracy']:.3f} | {m['precision_crisis']:.3f} | {m['recall_crisis']:.3f} | {m['f1_macro']:.3f} |"


b = metrics["binary"]
binary_card = model_card("CrisisConnect binary crisis classifier", f"""
Classifies a short text as `Crisis` or `Non-Crisis`. Fine-tuned from `{BASE_MODEL}` on project tweets, the Kaggle
*Disaster Tweets* (2020) dataset and TREC Incident Streams ({b['train_size']} training tweets; the best epoch was
chosen on a separate validation split).

| Test set | Tweets | Accuracy | Crisis precision | Crisis recall | Macro F1 |
|---|---|---|---|---|---|
{row("NLP with Disaster Tweets (2015), external", b["external_kaggle_2015"])}
{row("TREC-IS, unseen events", b["test_trecis_unseen_events"])}
{row("Project tweets, held out", b["test_project"])}

The external set was never used for training or model selection, and tweets overlapping the training data were
removed from every test set. Classifying a text does not verify it: the model detects crisis *language*, not
whether an event is real.
""")

p, pt = metrics["priority"], metrics["priority"]["test_trecis_unseen_events"]
priority_card = model_card("CrisisConnect crisis priority classifier", f"""
Classifies a crisis-related text as `Low`, `Medium`, `High` or `Critical` information priority, following the
[TREC Incident Streams](https://www.dcs.gla.ac.uk/~richardm/TREC_IS/) definition: how urgently an emergency manager
needs to see it. Fine-tuned from `{BASE_MODEL}` on {p['train_size']} TREC-IS tweets.

On TREC-IS events held out from training ({pt['n']} tweets): accuracy {pt['accuracy']:.3f}, macro F1
{pt['f1_macro']:.3f}, off by two or more levels {pt['off_by_2plus']:.3f}. For "needs prompt attention"
(High or Critical): precision {pt['urgent_precision']:.3f}, recall {pt['urgent_recall']:.3f}.

Priority is subjective and TREC-IS annotations are noisy, so treat outputs as triage hints for a human reviewer.
""")

for name, model, card in [("binary", binary_model, binary_card), ("priority", priority_model, priority_card)]:
    path = os.path.join(OUTPUT_DIR, name)
    model.save_pretrained(path)
    tokenizer.save_pretrained(path)
    with open(os.path.join(path, "README.md"), "w") as f:
        f.write(card)

with open(os.path.join(OUTPUT_DIR, "metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)
for name in ("binary", "priority"):
    shutil.copy(os.path.join(OUTPUT_DIR, "metrics.json"), os.path.join(OUTPUT_DIR, name, "metrics.json"))
print("saved to", OUTPUT_DIR)

## Upload

Uploads to **new** repos so the deployed models keep working until you choose to switch. Set `PUSH_TO_HUB = True`
and run; you will be asked for a Hugging Face token with write access.

To download instead, the second cell zips `output/` (about 1 GB).

In [ ]:
PUSH_TO_HUB = False
HUB_REPOS = {"binary": "ron4444444/crisis-binary-model-v2", "priority": "ron4444444/crisis-priority-model"}

if PUSH_TO_HUB:
    from huggingface_hub import HfApi, login
    login()
    api = HfApi()
    for name, repo in HUB_REPOS.items():
        api.create_repo(repo, exist_ok=True)
        api.upload_folder(folder_path=os.path.join(OUTPUT_DIR, name), repo_id=repo)
        print("uploaded", f"https://huggingface.co/{repo}")

In [ ]:
DOWNLOAD_ZIP = False
if DOWNLOAD_ZIP:
    archive = shutil.make_archive("crisis_models", "zip", OUTPUT_DIR)
    if IN_COLAB:
        colab_files.download(archive)
    print(archive)